In [5]:
# Assignment instructions https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/lab_theia_plotly_dash.md.html

# Steps to setup development environment
!pip install pandas dash
!pip install python3-wget


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Created wheel for python3-wget: filename=python3_wget-0.0.2b1-py3-none-any.whl size=7996 sha256=7e6dbacea7eb1020987771f12627ea288d305a04bd6762f6ee4a53176f138067
  Stored in directory: /Users/Tanoshi/Library/Caches/pip/wheels/2d/f3/eb/95aaad0d57e38bca665dd96a82037cc43dc4dbde5659d13e70
  Created wheel for hurry.filesize: filename=hurry.filesize-0.9-py3-none-any.whl size=4091 sha256=321ab59bcd3ae74e61b8acbcef89acee931aa9842a6e33af911cd945621a50b2
  Stored in directory: /Users/Tanoshi/Library/Caches/pip/wheels/c9/e5/86/2502e0cc50eeac1386815ab1c5e97640e9f83f58a6b88fa0ce
Successfully built python3-wget hurry.filesize


In [7]:
# Import wgte and required libraries, download the working dataset from course site 
import wget
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_dash.csv"
spacex_launch_dash = wget.download(url)

100% [############################################################]     2K / 2K

In [11]:
# Import required libraries
import pandas as pd
import dash
import dash_html_components as html
import dash_core_components as dcc
from dash.dependencies import Input, Output
import plotly.express as px

# Read the airline data into pandas dataframe
spacex_df = pd.read_csv("spacex_launch_dash.csv")
max_payload = spacex_df['Payload Mass (kg)'].max()
min_payload = spacex_df['Payload Mass (kg)'].min()

# Create a dash application
app = dash.Dash(__name__)

# Create an app layout
app.layout = html.Div(children=[html.H1('SpaceX Launch Records Dashboard',
                                        style={'textAlign': 'center', 'color': '#503D36',
                                               'font-size': 40}),
# TASK 1: Add a dropdown list to enable Launch Site selection
# The default select value is for ALL sites
# dcc.Dropdown(id='site-dropdown',...)
                                dcc.Dropdown(id='site-dropdown',
                                             options=[
                                                     {'label': 'All Sites', 'value': 'ALL'},
                                                     {'label': 'CCAFS LC-40', 'value': 'CCAFS LC-40'},
                                                     {'label': 'VAFB SLC-4E', 'value': 'VAFB SLC-4E'},
                                                     {'label': 'KSC LC-39A', 'value': 'KSC LC-39A'},
                                                     {'label': 'CCAFS SLC-40', 'value': 'CCAFS SLC-40'}
                                                     ],
                                             value='ALL',
                                             placeholder='Select a Launch Site here',
                                             searchable=True,
                                             style={'width':'80%','padding':'3px','font-size':'20px','text-align-last':'center'}
                                             ),
                                html.Br(),

# TASK 2: Add a pie chart to show the total successful launches count for all sites
# If a specific launch site was selected, show the Success vs. Failed counts for the site
                                html.Div(dcc.Graph(id='success-pie-chart')),
                                html.Br(),

                                html.P("Payload range (Kg):"),
# TASK 3: Add a slider to select payload range
#dcc.RangeSlider(id='payload-slider',...)
                                dcc.RangeSlider(id='payload-slider',
                                                min=0,
                                                max=10000,
                                                step=1000,
                                                value=[min_payload, max_payload]
                                                ),

# TASK 4: Add a scatter chart to show the correlation between payload and launch success
                                html.Div(dcc.Graph(id='success-payload-scatter-chart')),
                                ])

# TASK 2:
# Add a callback function for `site-dropdown` as input, `success-pie-chart` as output
@app.callback(Output(component_id='success-pie-chart', component_property='figure'),
              Input(component_id='site-dropdown', component_property='value'))
def get_pie_chart(ssite):
    fnew_df = spacex_df
    if ssite == 'ALL':
        graph = px.pie(fnew_df, values='class', 
        names='Launch Site', 
        title='Success Count for all launch sites')
        return graph
    else:
# return the outcomes piechart for a selected site
        fnew_df=spacex_df[spacex_df['Launch Site']== ssite]
        fnew_df=fnew_df.groupby(['Launch Site','class']).size().reset_index(name='class count')
        graph=px.pie(fnew_df,values='class count',names='class',title=f"Total Success Launches for site {ssite}")
        return graph

# TASK 4:
# Add a callback function
@app.callback(Output(component_id='success-payload-scatter-chart',component_property='figure'),
                [Input(component_id='site-dropdown',component_property='value'),
                Input(component_id='payload-slider',component_property='value')])
def scatter(ssite,payload):
    fnew_df = spacex_df[spacex_df['Payload Mass (kg)'].between(payload[0],payload[1])]    
    if ssite=='ALL':
        graph=px.scatter(fnew_df,x='Payload Mass (kg)',y='class',color='Booster Version Category',title='Success count on Payload mass for all sites')
        return graph
    else:
        graph=px.scatter(fnew_df[fnew_df['Launch Site']==ssite],x='Payload Mass (kg)',y='class',color='Booster Version Category',title=f"Success count on Payload mass for site {ssite}")
        return graph
if __name__ == '__main__':
    app.run_server()